# STORM: Structural & Tandem-Repeat Optimized Regression Models

This notebook demonstrates end-to-end usage of STORM for association testing of structural variants and tandem repeats.

## 1. Setup and Import

First, ensure the storm package is available. If running from the repo, add the dev path.

In [1]:
import sys
import subprocess
from pathlib import Path

# Add development path if running from repo
sys.path.insert(0, "../python")

import storm
import polars as pl

print(f"STORM version: {storm.version()}")
print(f"Rust bindings available: {storm.HAS_RUST}")

STORM version: 0.0.23
Rust bindings available: True


In [2]:
# Check available fixtures
fixtures = Path("../fixtures")
print("Available fixtures:")
for f in sorted(fixtures.glob("*")):
    print(f"  {f.name}: {f.stat().st_size} bytes")

Available fixtures:
  plan.yaml: 483 bytes
  sv_small.vcf: 611 bytes
  trexplorer.bed: 205 bytes
  trexplorer.json: 1227 bytes
  trgt_sample1.vcf: 883 bytes
  trgt_sample2.vcf: 882 bytes
  trgt_small.vcf: 1075 bytes


## 2. Build Cache from Input Files

STORM builds a cache from:
- Integrated SV VCF (structural variants)
- TRGT VCF (tandem repeat genotypes)
- TRExplorer BED/JSON catalog

In [3]:
# Build cache from fixtures
cache_dir = "demo_cache"

cache = storm.StormCache.build(
    sv_vcf="../fixtures/sv_small.vcf",
    trgt_vcf="../fixtures/trgt_small.vcf",
    catalog_bed="../fixtures/trexplorer.bed",
    catalog_json="../fixtures/trexplorer.json",
    output_dir=cache_dir,
)

print(f"\nCache built at: {cache_dir}")
print(f"Build stats: {getattr(cache, '_build_stats', 'N/A')}")


Cache built at: demo_cache
Build stats: {'num_test_units': 6, 'num_samples': 2, 'num_genotypes': 12, 'num_catalog_entries': 4}


## 3. Load and Explore Cache

The cache stores data in Parquet format, which Polars can load efficiently.

In [4]:
# Load cache (or use existing cache object)
cache = storm.load_cache(cache_dir)

# View test units
print("Test Units:")
print(cache.test_units)

Test Units:
shape: (6, 8)
┌──────────────┬───────┬───────┬───────┬────────────┬─────────┬────────────┬───────┐
│ id           ┆ chrom ┆ start ┆ end   ┆ unit_type  ┆ source  ┆ catalog_id ┆ motif │
│ ---          ┆ ---   ┆ ---   ┆ ---   ┆ ---        ┆ ---     ┆ ---        ┆ ---   │
│ str          ┆ str   ┆ u64   ┆ u64   ┆ str        ┆ str     ┆ str        ┆ str   │
╞══════════════╪═══════╪═══════╪═══════╪════════════╪═════════╪════════════╪═══════╡
│ sv_sv1       ┆ chr1  ┆ 1000  ┆ 1500  ┆ Sv         ┆ SvVcf   ┆ null       ┆ null  │
│ sv_sv2       ┆ chr1  ┆ 2000  ┆ 2200  ┆ Sv         ┆ SvVcf   ┆ null       ┆ null  │
│ sv_sv3       ┆ chr2  ┆ 5000  ┆ 6000  ┆ Sv         ┆ SvVcf   ┆ null       ┆ null  │
│ repeat_TR001 ┆ chr1  ┆ 10000 ┆ 10009 ┆ TrueRepeat ┆ TrgtVcf ┆ TR001      ┆ CAG   │
│ repeat_TR002 ┆ chr2  ┆ 20000 ┆ 20004 ┆ TrueRepeat ┆ TrgtVcf ┆ TR002      ┆ AT    │
│ repeat_TR003 ┆ chr3  ┆ 30000 ┆ 30009 ┆ TrueRepeat ┆ TrgtVcf ┆ TR003      ┆ GGC   │
└──────────────┴───────┴───────┴───────

In [5]:
# View genotypes
print("Genotypes:")
print(cache.genotypes)

Genotypes:
shape: (12, 7)
┌──────────────┬───────────┬────────────┬─────────────────┬─────────┬─────────┬───────────────┐
│ unit_id      ┆ sample_id ┆ is_present ┆ presence_source ┆ allele1 ┆ allele2 ┆ allele_source │
│ ---          ┆ ---       ┆ ---        ┆ ---             ┆ ---     ┆ ---     ┆ ---           │
│ str          ┆ str       ┆ bool       ┆ str             ┆ i64     ┆ i64     ┆ str           │
╞══════════════╪═══════════╪════════════╪═════════════════╪═════════╪═════════╪═══════════════╡
│ sv_sv1       ┆ SAMPLE1   ┆ true       ┆ SvVcf           ┆ 0       ┆ 500     ┆ SvProxy       │
│ sv_sv1       ┆ SAMPLE2   ┆ true       ┆ SvVcf           ┆ 500     ┆ 500     ┆ SvProxy       │
│ sv_sv2       ┆ SAMPLE1   ┆ true       ┆ SvVcf           ┆ 0       ┆ 200     ┆ SvProxy       │
│ sv_sv2       ┆ SAMPLE2   ┆ false      ┆ SvVcf           ┆ 0       ┆ 0       ┆ SvProxy       │
│ sv_sv3       ┆ SAMPLE1   ┆ false      ┆ SvVcf           ┆ 0       ┆ 0       ┆ SvProxy       │
│ …           

In [6]:
# View catalog
print("Catalog:")
print(cache.catalog)

Catalog:
shape: (4, 10)
┌───────┬───────┬───────┬───────┬───┬──────┬───────────┬────────────┬────────────────┐
│ id    ┆ chrom ┆ start ┆ end   ┆ … ┆ gene ┆ disease   ┆ normal_max ┆ pathogenic_min │
│ ---   ┆ ---   ┆ ---   ┆ ---   ┆   ┆ ---  ┆ ---       ┆ ---        ┆ ---            │
│ str   ┆ str   ┆ u64   ┆ u64   ┆   ┆ str  ┆ str       ┆ u64        ┆ u64            │
╞═══════╪═══════╪═══════╪═══════╪═══╪══════╪═══════════╪════════════╪════════════════╡
│ TR001 ┆ chr1  ┆ 10000 ┆ 10050 ┆ … ┆ HTT  ┆ null      ┆ 26         ┆ 36             │
│ TR003 ┆ chr2  ┆ 20000 ┆ 20030 ┆ … ┆ FMR1 ┆ Fragile X ┆ 44         ┆ 200            │
│ TR004 ┆ chr3  ┆ 30000 ┆ 30060 ┆ … ┆ null ┆ null      ┆ null       ┆ null           │
│ TR002 ┆ chr1  ┆ 50000 ┆ 50100 ┆ … ┆ null ┆ null      ┆ null       ┆ null           │
└───────┴───────┴───────┴───────┴───┴──────┴───────────┴────────────┴────────────────┘


In [7]:
# View features
print("Features:")
print(cache.features)

Features:
shape: (12, 6)
┌──────────────┬───────────┬───────┬───────┬────────┬────────┐
│ unit_id      ┆ sample_id ┆ D     ┆ M     ┆ S      ┆ binary │
│ ---          ┆ ---       ┆ ---   ┆ ---   ┆ ---    ┆ ---    │
│ str          ┆ str       ┆ f64   ┆ f64   ┆ f64    ┆ f64    │
╞══════════════╪═══════════╪═══════╪═══════╪════════╪════════╡
│ sv_sv1       ┆ SAMPLE1   ┆ 500.0 ┆ 500.0 ┆ 500.0  ┆ 1.0    │
│ sv_sv1       ┆ SAMPLE2   ┆ 0.0   ┆ 500.0 ┆ 1000.0 ┆ 1.0    │
│ sv_sv2       ┆ SAMPLE1   ┆ 200.0 ┆ 200.0 ┆ 200.0  ┆ 1.0    │
│ sv_sv2       ┆ SAMPLE2   ┆ 0.0   ┆ 0.0   ┆ 0.0    ┆ 0.0    │
│ sv_sv3       ┆ SAMPLE1   ┆ 0.0   ┆ 0.0   ┆ 0.0    ┆ 0.0    │
│ …            ┆ …         ┆ …     ┆ …     ┆ …      ┆ …      │
│ repeat_TR001 ┆ SAMPLE2   ┆ 3.0   ┆ 15.0  ┆ 27.0   ┆ 1.0    │
│ repeat_TR002 ┆ SAMPLE1   ┆ 0.0   ┆ 4.0   ┆ 8.0    ┆ 1.0    │
│ repeat_TR002 ┆ SAMPLE2   ┆ 2.0   ┆ 6.0   ┆ 10.0   ┆ 1.0    │
│ repeat_TR003 ┆ SAMPLE1   ┆ 0.0   ┆ 12.0  ┆ 24.0   ┆ 1.0    │
│ repeat_TR003 ┆ SAMPLE2   ┆ 3

## 4. Explain Genotypes

Get detailed information about resolved genotypes at a locus.

In [8]:
# Get available test unit IDs
unit_ids = cache.test_units["id"].to_list()
print(f"Available unit IDs: {unit_ids}")

Available unit IDs: ['sv_sv1', 'sv_sv2', 'sv_sv3', 'repeat_TR001', 'repeat_TR002', 'repeat_TR003']


In [9]:
# Explain a locus (all samples)
unit_id = unit_ids[0]
explanation = storm.explain(cache, unit_id)
print(explanation)

=== Locus Summary ===

Test Unit: sv_sv1
Location: chr1:1000-1500
Type: Sv

Summary Statistics:
  Total samples: 2
  Present (carriers): 2
  Absent (non-carriers): 0
  Call rate: 100.00%

Sample Details:
Sample           Present    Allele1    Allele2       Source
------------------------------------------------------------
SAMPLE1              Yes          0        500      SvProxy
SAMPLE2              Yes        500        500      SvProxy



In [10]:
# Explain for a specific sample
explanation = storm.explain(cache, unit_id, sample_id="SAMPLE1")
print(explanation)

=== Genotype Explanation ===

Test Unit: sv_sv1
  Location: chr1:1000-1500
  Type: Sv

Sample: SAMPLE1
  Presence: PRESENT

Alleles:
  Allele 1: 0 bp
  Allele 2: 500 bp
  Source: SvProxy

Computed Encodings:
  S (sum): 500 bp
  M (max): 500 bp
  D (diff): 500 bp



## 5. Run Association Testing

STORM supports multiple models:
- Linear regression for quantitative traits
- Logistic regression for binary traits
- BinomiRare for rare variant burden testing
- Firth regression for rare events

In [11]:
# Note: The demo fixtures have only 2 samples, which is below the minimum
# of 3 required for regression. In real data with more samples, you would
# see actual beta, SE, and p-values.

# Create a dummy phenotype
n_samples = 2
phenotype = pl.Series("phenotype", [0.5, 1.5])

# Run association
# With 2 samples, regression cannot run - error field shows "Insufficient samples"
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Rust run_association failed")
    results = storm.run_glm(
        cache=cache,
        phenotype=phenotype,
    )

print("Association Results (with 2-sample demo data):")
print(results)
print("\n(Note: beta=0, se=1, p_value=1 are fallback values when n_samples < 3)")

Association Results (with 2-sample demo data):
shape: (6, 11)
┌──────┬───────────┬──────────┬────────────────────┬───┬─────────┬──────┬───────────┬──────────────┐
│ beta ┆ call_rate ┆ encoding ┆ error              ┆ … ┆ p_value ┆ se   ┆ statistic ┆ unit_id      │
│ ---  ┆ ---       ┆ ---      ┆ ---                ┆   ┆ ---     ┆ ---  ┆ ---       ┆ ---          │
│ null ┆ f64       ┆ str      ┆ str                ┆   ┆ null    ┆ null ┆ null      ┆ str          │
╞══════╪═══════════╪══════════╪════════════════════╪═══╪═════════╪══════╪═══════════╪══════════════╡
│ null ┆ 1.0       ┆ S        ┆ Insufficient       ┆ … ┆ null    ┆ null ┆ null      ┆ sv_sv1       │
│      ┆           ┆          ┆ samples for regre… ┆   ┆         ┆      ┆           ┆              │
│ null ┆ 1.0       ┆ S        ┆ Insufficient       ┆ … ┆ null    ┆ null ┆ null      ┆ sv_sv2       │
│      ┆           ┆          ┆ samples for regre… ┆   ┆         ┆      ┆           ┆              │
│ null ┆ 1.0       ┆ S       

## 6. Cache Verification

Verify the cache structure and integrity.

In [12]:
# Verify cache
result = storm.verify_cache(cache_dir)
print("Cache verification result:")
for key, value in result.items():
    print(f"  {key}: {value}")

Cache verification result:
  is_valid: True
  num_test_units: 6
  num_genotypes: 12
  num_catalog_entries: 4
  num_features: 12
  errors: []


## 7. CLI Usage (Alternative)

STORM also provides a CLI for command-line workflows.

In [13]:
# Build cache via CLI
!cd .. && cargo run --release -- cache build \
    --sv-vcf fixtures/sv_small.vcf \
    --trgt-vcf fixtures/trgt_small.vcf \
    --catalog-bed fixtures/trexplorer.bed \
    --catalog-json fixtures/trexplorer.json \
    --output-dir /tmp/cli_cache

   Compiling libc v0.2.172
   Compiling once_cell v1.21.3
   Compiling syn v2.0.101
   Compiling getrandom v0.3.4==>    ] 153/181: libc, syn                   
   Compiling cpufeatures v0.2.17
   Compiling sha2 v0.10.9======>    ] 154/181: getrandom, syn, cpufeatures 
   Compiling ahash v0.8.12=====>    ] 155/181: sha2, getrandom, syn        
   Compiling arrow-array v54.3.1    ] 156/181: sha2, syn, ahash            
   Compiling serde_derive v1.0.228  ] 157/181: sha2, arrow-array, syn      
   Compiling thiserror-impl v2.0.12
   Compiling clap_derive v4.5.32
   Compiling thiserror v2.0.12=>    ] 159/181: thiserror-impl, arrow-arr...
   Compiling clap v4.5.39=======>   ] 161/181: arrow-array, clap_derive,...
   Compiling serde v1.0.228=====>   ] 163/181: arrow-array, serde_derive   
   Compiling arrow-select v54.3.1   ] 164/181: serde, arrow-array          
   Compiling arrow-ipc v54.3.1
   Compiling arrow-row v54.3.1
   Compiling arrow-arith v54.3.1
   Compiling serde_json v1.0.140>  

In [14]:
# Verify cache via CLI
!cd .. && cargo run --release -- cache verify --cache-dir /tmp/cli_cache

    Finished ]8;;https://doc.rust-lang.org/cargo/reference/profiles.html#default-profiles\`release` profile [optimized]]8;;\ target(s) in 0.21s
     Running `target/release/storm cache verify --cache-dir /tmp/cli_cache`
Verifying STORM cache at /tmp/cli_cache...

✓ Cache is valid!
  Test units: 6
  Genotypes: 12
  Catalog entries: 4
  Features: 12


In [15]:
# Explain via CLI
!cd .. && cargo run --release -- explain sv_sv1 --cache-dir /tmp/cli_cache

    Finished ]8;;https://doc.rust-lang.org/cargo/reference/profiles.html#default-profiles\`release` profile [optimized]]8;;\ target(s) in 0.10s
     Running `target/release/storm explain sv_sv1 --cache-dir /tmp/cli_cache`
=== Locus Summary ===

Test Unit: sv_sv1
Location: chr1:1000-1500
Type: Sv

Summary Statistics:
  Total samples: 2
  Present (carriers): 2
  Absent (non-carriers): 0
  Call rate: 100.00%

Sample Details:
Sample           Present    Allele1    Allele2       Source
------------------------------------------------------------
SAMPLE1              Yes          0        500      SvProxy
SAMPLE2              Yes        500        500      SvProxy



## Summary

STORM provides a complete workflow for:

1. **Parsing** - SV VCF, TRGT VCF, TRExplorer catalogs
2. **Resolution** - Mapping SVs to repeat loci, reconstructing diploid lengths
3. **Caching** - Arrow/Parquet storage for efficient access
4. **Analysis** - Multiple regression models and encodings
5. **Output** - Parquet results with full metadata

The Rust core ensures correctness and performance, while the Python API provides interactive exploration via Polars.

---

## 8. Real-Data Association Testing Workflow

The sections above demonstrate STORM with small fixture files (2 samples). Below we show a **full association testing workflow** using real development data:

- **SV BCF:** `scratch/chr6.31803187_32050925.bcf` (~12,680 samples)
- **TRGT VCFs:** `scratch/trgt/*.vcf.gz` (~10,003 per-sample files)

This demonstrates how to run meaningful statistical tests with large sample sizes.

In [ ]:
# Check if real data is available
scratch_dir = Path("../scratch")
sv_bcf = scratch_dir / "chr6.31803187_32050925.bcf"
trgt_dir = scratch_dir / "trgt"

HAS_REAL_DATA = sv_bcf.exists() and trgt_dir.exists()

if HAS_REAL_DATA:
    trgt_files = sorted(trgt_dir.glob("*.vcf.gz"))
    print(f"✓ Real data found!")
    print(f"  SV BCF: {sv_bcf} ({sv_bcf.stat().st_size / 1e6:.1f} MB)")
    print(f"  TRGT files: {len(trgt_files)} VCF.gz files in {trgt_dir}")
else:
    print("⚠ Real data not found in scratch/ - skipping real-data sections.")
    print("  To run these sections, download the development data (see DEV_DATA.md).")

### 8.1 Build Cache from Real Data

We use a subset of TRGT files for demo speed. Adjust `N_TRGT_FILES` to scale up:

| TRGT files | Approx runtime | Use case |
|------------|----------------|----------|
| 100        | ~10 seconds    | Quick demo |
| 500        | ~30 seconds    | Moderate analysis |
| All (~10k) | ~5-10 minutes  | Full cohort |

In [ ]:
if HAS_REAL_DATA:
    import time
    
    # Configurable: number of TRGT files to process
    N_TRGT_FILES = 200  # Use 200 for a good balance of speed and sample size
    
    # Select subset of TRGT files
    trgt_subset = [str(f) for f in trgt_files[:N_TRGT_FILES]]
    print(f"Building cache with {N_TRGT_FILES} TRGT files...")
    
    # Build cache
    real_cache_dir = "real_data_cache"
    start_time = time.time()
    
    real_cache = storm.StormCache.build(
        sv_vcf=str(sv_bcf),
        trgt_vcf=trgt_subset,
        output_dir=real_cache_dir,
    )
    
    elapsed = time.time() - start_time
    print(f"\n✓ Cache built in {elapsed:.1f} seconds")
    
    # Show build statistics
    stats = getattr(real_cache, '_build_stats', {})
    print(f"\nCache Statistics:")
    print(f"  Test units: {stats.get('num_test_units', 'N/A')}")
    print(f"  Samples: {stats.get('num_samples', 'N/A')}")
    print(f"  Genotypes: {stats.get('num_genotypes', 'N/A')}")
else:
    print("Skipping - real data not available")

In [ ]:
if HAS_REAL_DATA:
    # View sample list
    sample_ids = real_cache.genotypes["sample_id"].unique().sort().to_list()
    print(f"Samples in cache: {len(sample_ids)}")
    print(f"First 10 samples: {sample_ids[:10]}")
    
    # View test units summary
    print(f"\nTest Units by type:")
    print(real_cache.test_units.group_by("unit_type").agg(pl.count().alias("count")))

### 8.2 Generate Simulated Phenotypes

For demonstration, we generate simulated phenotypes. In practice, you would load real phenotype data from a file.

We create:
1. **Continuous phenotype** (e.g., height, BMI) - for linear regression
2. **Binary phenotype** (case/control) - for logistic regression
3. **Covariates** (age, sex, PCs) - for adjusted models

In [ ]:
if HAS_REAL_DATA:
    import numpy as np
    
    # Set seed for reproducibility
    np.random.seed(42)
    n_samples = len(sample_ids)
    
    # 1. Continuous phenotype (simulated trait, e.g., normalized height)
    continuous_phenotype = pl.Series("phenotype", np.random.randn(n_samples))
    
    # 2. Binary phenotype (case/control with ~30% cases)
    binary_phenotype = pl.Series("phenotype", np.random.binomial(1, 0.3, n_samples).astype(float))
    
    # 3. Covariates (age, sex, first 3 PCs)
    covariates = pl.DataFrame({
        "sample_id": sample_ids,
        "age": np.random.normal(50, 15, n_samples),  # Mean age 50, SD 15
        "sex": np.random.binomial(1, 0.5, n_samples),  # 0=female, 1=male
        "PC1": np.random.randn(n_samples),
        "PC2": np.random.randn(n_samples),
        "PC3": np.random.randn(n_samples),
    })
    
    print(f"Generated phenotypes for {n_samples} samples:")
    print(f"  Continuous: mean={continuous_phenotype.mean():.3f}, std={continuous_phenotype.std():.3f}")
    print(f"  Binary: {binary_phenotype.sum():.0f} cases ({100*binary_phenotype.mean():.1f}%)")
    print(f"\nCovariates preview:")
    print(covariates.head())
else:
    print("Skipping - real data not available")

In [ ]:
# Example: Loading phenotypes from external files (not executed)
# 
# From CSV:
#   pheno_df = pl.read_csv("phenotypes.csv")
#   phenotype = pheno_df.filter(pl.col("sample_id").is_in(sample_ids))["phenotype"]
#
# From Parquet:
#   pheno_df = pl.read_parquet("phenotypes.parquet")
#   phenotype = pheno_df.filter(pl.col("sample_id").is_in(sample_ids))["phenotype"]
#
# Important: Ensure phenotype order matches sample_ids order from the cache!

print("See comments above for loading phenotypes from CSV/Parquet files.")

### 8.3 Run Association Tests

Now we run association testing with the real-data cache. With hundreds of samples, we get actual statistical results (beta, SE, p-values) instead of the "Insufficient samples" errors from the 2-sample fixtures.

In [ ]:
if HAS_REAL_DATA:
    # Linear regression with continuous phenotype
    print("Running linear regression (continuous phenotype)...")
    linear_results = storm.run_glm(
        cache=real_cache,
        phenotype=continuous_phenotype,
        model="linear",
        encoding="S",  # Sum encoding (allele1 + allele2)
    )
    
    print(f"\nLinear regression results: {len(linear_results)} test units")
    print(linear_results.select(["unit_id", "beta", "se", "p_value", "n_samples", "encoding", "model"]))
else:
    print("Skipping - real data not available")

In [16]:
# Cleanup
import shutil
shutil.rmtree(cache_dir, ignore_errors=True)
print("Demo cache cleaned up.")

Demo cache cleaned up.
